# PrimePE — Phase 3: Real Language Modelling (H100 Edition)

> **Version:** v0.3.1-h100 | **Author:** Knack | **Date:** 2026-03-27  
> **Target:** Colab Pro H100 (80GB HBM3) — bf16 + FlashAttention-2 + torch.compile

## What this runs
| Step | Detail |
|------|--------|
| Model | 12-layer, d=512, 16 heads — ~125M params (GPT-2 medium scale) |
| Data | WikiText-103 (real language, not synthetic) |
| Training | 20K steps, bf16, FlashAttention-2, torch.compile |
| Eval seq lens | 512 / 2K / 4K / 8K / **16K** / **32K** |
| LitM context | 8K tokens — where RoPE aliasing actually bites |
| PE variants | sinusoidal, rope, zeta, hybrid_90z, hybrid_50z, prime_05, random_irr, learned, alibi |

## Why H100 unlocks the real test
- 80GB HBM3 → can hold a 125M model + 32K context batch comfortably in bf16
- FlashAttention-2 → O(N) memory for long contexts, no OOM at 32K
- torch.compile → ~40% throughput gain, brings total runtime to ~2-3hrs for full suite
- 32K context → **this is where sinusoidal PE aliases and ZetaPE shouldn't**

---
**Runtime estimate:** ~2.5–3.5 hrs for full 9-variant suite.  
Set `QUICK_RUN = True` in Cell 3 for a 3000-step smoke test (~25 min).


In [ ]:
# ─── Cell 1: Install ─────────────────────────────────────────────────────────
# PyTorch SDPA auto-routes to FlashAttention-2 on H100+bf16 — no compile needed.
!pip install -q datasets transformers tokenizers matplotlib seaborn
print("Done.")


In [ ]:
# ─── Cell 2: GPU check ─────────────────────────────────────────────────────────
import torch, math, os, json, time, gc
import numpy as np

assert torch.cuda.is_available(), "No GPU — check runtime type"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU:  {gpu}")
print(f"VRAM: {vram:.0f} GB")

IS_H100 = "H100" in gpu or "A100" in gpu
DTYPE   = torch.bfloat16 if IS_H100 else torch.float16
print(f"dtype: {DTYPE} | FlashAttn: {'yes' if IS_H100 else 'limited'}")

# PyTorch SDPA auto-selects FlashAttention-2 kernel on H100+bf16
FLASH_AVAILABLE = False  # not needed — SDPA handles it
try:
    with torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.FLASH_ATTENTION):
        pass
    print("FlashAttention-2 via SDPA: available ✅")
except Exception:
    print("SDPA fallback mode")

DEVICE = torch.device("cuda")

In [ ]:
# ─── Cell 3: Mission Control ─────────────────────────────────────────────────
# All tuneable values live here. GPU profiles auto-set defaults.
# Version: v0.3.3 | Author: Knack

# ── Override flags — set these manually if needed ────────────────────────────
QUICK_RUN    = False   # True = 3K steps, short evals, ~25 min smoke test
FORCE_SMALL  = False   # True = force small model (overrides GPU profile)

# ── GPU auto-profile ─────────────────────────────────────────────────────────
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
gpu_name = torch.cuda.get_device_name(0)

if "H100" in gpu_name or "A100" in gpu_name:
    GPU_PROFILE = "H100"   # 80GB — full config
elif "A40" in gpu_name or "A30" in gpu_name or vram_gb >= 30:
    GPU_PROFILE = "A40"    # 40-48GB — medium config
elif "T4" in gpu_name or "V100" in gpu_name or vram_gb >= 12:
    GPU_PROFILE = "T4"     # 12-16GB — scaled down
else:
    GPU_PROFILE = "T4"     # fallback conservative

if FORCE_SMALL:
    GPU_PROFILE = "T4"

print(f"GPU: {gpu_name}  ({vram_gb:.0f}GB)  → profile: {GPU_PROFILE}")

# ── Per-profile configs ───────────────────────────────────────────────────────
PROFILES = {
    "H100": {
        "d_model": 512, "n_heads": 16, "n_layers": 12, "ffn_dim": 2048,
        "max_steps": 3_000 if QUICK_RUN else 20_000,
        "batch_size": 4 if QUICK_RUN else 8,
        "grad_accum": 4,
        "warmup_steps": 200 if QUICK_RUN else 1_000,
        "eval_seq_lens": [512, 1024, 2048, 4096, 8192] if QUICK_RUN
                         else [512, 1024, 2048, 4096, 8192, 16384, 32768],
        "litm_context_len": 2048 if QUICK_RUN else 8192,
        "litm_samples":     100  if QUICK_RUN else 500,
        "use_compile": True,
    },
    "A40": {
        "d_model": 512, "n_heads": 16, "n_layers": 12, "ffn_dim": 2048,
        "max_steps": 3_000 if QUICK_RUN else 20_000,
        "batch_size": 2 if QUICK_RUN else 4,
        "grad_accum": 8,
        "warmup_steps": 200 if QUICK_RUN else 1_000,
        "eval_seq_lens": [512, 1024, 2048, 4096, 8192],
        "litm_context_len": 2048 if QUICK_RUN else 8192,
        "litm_samples":     100  if QUICK_RUN else 300,
        "use_compile": True,
    },
    "T4": {
        "d_model": 512, "n_heads": 16, "n_layers": 12, "ffn_dim": 2048,
        "max_steps": 3_000 if QUICK_RUN else 10_000,
        "batch_size": 1 if QUICK_RUN else 2,
        "grad_accum": 16,  # effective batch = 32 regardless
        "warmup_steps": 200 if QUICK_RUN else 500,
        "eval_seq_lens": [512, 1024, 2048, 4096] if QUICK_RUN
                         else [512, 1024, 2048, 4096, 8192],
        "litm_context_len": 2048 if QUICK_RUN else 4096,
        "litm_samples":     50   if QUICK_RUN else 200,
        "use_compile": True,
    },
}

p = PROFILES[GPU_PROFILE]

CFG = {
    # ── Model ─────────────────────────────────────────────────────────────────
    "d_model":      p["d_model"],
    "n_heads":      p["n_heads"],
    "n_layers":     p["n_layers"],
    "ffn_dim":      p["ffn_dim"],
    "dropout":      0.1,
    "vocab_size":   50257,

    # ── Training ──────────────────────────────────────────────────────────────
    "max_steps":    p["max_steps"],
    "batch_size":   p["batch_size"],
    "grad_accum":   p["grad_accum"],
    "lr":           3e-4,
    "warmup_steps": p["warmup_steps"],
    "weight_decay": 0.01,
    "clip_grad":    1.0,
    "eval_every":   500,
    "train_seq_len": 1024,

    # ── Eval ──────────────────────────────────────────────────────────────────
    "eval_seq_lens":    p["eval_seq_lens"],
    "litm_context_len": p["litm_context_len"],
    "litm_positions":   [0.1, 0.25, 0.5, 0.75, 0.9],
    "litm_samples":     p["litm_samples"],

    # ── PE variants ───────────────────────────────────────────────────────────
    "pe_variants": [
        "sinusoidal",      # additive baseline
        "rope",            # rotary baseline
        "zeta_rope",       # KEY: zeta freqs in RoPE arch — apples-to-apples
        "random_irr_rope", # CONTROL: random irrational in RoPE arch
        "zeta",            # additive zeta
        "hybrid_90z",      # additive 90% zeta / 10% prime
        "random_irr",      # CONTROL: additive random irrational
        "learned",         # CONTROL: geometric init trained
        "alibi",           # length-generalisation baseline
    ],

    # ── Runtime ───────────────────────────────────────────────────────────────
    "use_compile":    p["use_compile"],
    "use_flash_attn": False,   # SDPA handles this automatically
    "dtype":          DTYPE,
    "gpu_profile":    GPU_PROFILE,

    # ── Output ────────────────────────────────────────────────────────────────
    "results_dir": "/content/results",
    "seed":        42,
}

import os
os.makedirs(CFG["results_dir"], exist_ok=True)

eff_batch = CFG["batch_size"] * CFG["grad_accum"]
print(f"Profile:        {GPU_PROFILE}  {'(QUICK)' if QUICK_RUN else ''}")
print(f"Model:          {CFG['n_layers']}L × d{CFG['d_model']} × {CFG['n_heads']}h")
print(f"Steps:          {CFG['max_steps']:,}")
print(f"Effective batch:{eff_batch}  ({CFG['batch_size']} × {CFG['grad_accum']} accum)")
print(f"Train seq len:  {CFG['train_seq_len']}")
print(f"Eval ctxs:      {CFG['eval_seq_lens']}")
print(f"LitM ctx:       {CFG['litm_context_len']} tokens × {CFG['litm_samples']} samples")
print(f"torch.compile:  {CFG['use_compile']}  (mode=default — no CUDAGraphs)")
print(f"PE variants:    {len(CFG['pe_variants'])} → {CFG['pe_variants']}")


In [ ]:
# ─── Cell 4: PE frequency definitions ─────────────────────────────────────────
# Version: v0.3.1-h100

import torch.nn as nn
from typing import Optional

# ─── Zeta zeros — extended list (128 values for d=512) ────────────────────────
ZETA_ZEROS = [
    14.134725, 21.022040, 25.010858, 30.424876, 32.935061,
    37.586178, 40.918719, 43.327073, 48.005150, 49.773832,
    52.970321, 56.446247, 59.347044, 60.831778, 65.112544,
    67.079810, 69.546401, 72.067157, 75.704691, 77.144840,
    79.337375, 82.910381, 84.735493, 87.425275, 88.809111,
    92.491899, 94.651344, 95.870634, 98.831194, 101.317851,
    103.725538, 105.446623, 107.168611, 111.029535, 111.874659,
    114.320220, 116.226680, 118.790782, 121.370125, 122.946829,
    124.256818, 127.516683, 129.578704, 131.087688, 133.497737,
    134.756509, 138.116042, 139.736209, 141.123707, 143.111845,
    146.000982, 147.422765, 150.053521, 150.925257, 153.024693,
    156.112909, 157.597591, 158.849988, 161.188964, 163.030709,
    165.537069, 167.184439, 169.094515, 169.911976, 173.411536,
    174.754191, 176.441434, 178.377407, 179.916484, 182.207078,
    184.874467, 185.598783, 187.228922, 189.416158, 192.026656,
    193.079726, 195.265396, 196.876481, 198.015309, 201.264751,
    202.493594, 204.189671, 205.394697, 207.906258, 209.576509,
    211.690862, 213.347919, 214.547044, 216.169538, 219.067596,
    220.714918, 221.430705, 224.007000, 224.983324, 227.421444,
    229.337413, 231.250188, 231.987235, 233.693404, 236.524229,
    237.769820, 239.555477, 241.049157, 242.823271, 244.070898,
    247.136990, 248.101990, 249.573369, 251.014944, 253.069859,
    255.306388, 256.380713, 258.610439, 259.874406, 260.805801,
    263.573893, 265.557536, 266.614973, 267.921918, 269.969904,
    271.494350, 273.459609, 275.587492, 276.452049, 278.251218,
    279.229250, 282.465124, 283.211019, 284.835963, 285.752021,
]

def get_primes(n: int) -> list:
    primes, c = [], 2
    while len(primes) < n:
        if all(c % p != 0 for p in primes):
            primes.append(c)
        c += 1
    return primes

def get_zeta_freqs(n: int) -> torch.Tensor:
    """Return n normalised zeta-zero frequencies, extending with approximation if needed."""
    zz = list(ZETA_ZEROS[:n])
    if len(zz) < n:
        for i in range(len(zz), n):
            k = float(i + 1)
            zz.append((2 * math.pi * k) / (math.log(k) + 0.5))
    t = torch.tensor(zz[:n], dtype=torch.float32)
    return t / t[0]  # normalise to [1, ...]

def freq_spec(pe_name: str, d_model: int) -> dict:
    """Returns frequency tensor + flags for a PE variant."""
    half = d_model // 2
    if pe_name == "sinusoidal":
        return {"type": "additive", "learnable": False,
                "freqs": torch.exp(-torch.arange(0, half).float() * math.log(10000.0) / half)}
    elif pe_name == "rope":
        return {"type": "rope", "learnable": False,
                "freqs": 1.0 / (10000 ** (torch.arange(0, half, 2).float() / half))}
    elif pe_name == "alibi":
        return {"type": "alibi", "learnable": False, "freqs": None}
    elif pe_name == "prime_05":
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        return {"type": "additive", "learnable": False, "freqs": 1.0 / (p ** 0.5)}
    elif pe_name == "prime_10":
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        return {"type": "additive", "learnable": False, "freqs": 1.0 / p}
    elif pe_name == "zeta":
        return {"type": "additive", "learnable": False, "freqs": get_zeta_freqs(half)}
    elif pe_name == "hybrid_90z":
        # v1.4.0: 90% zeta / 10% prime interleaved
        # Best absolute PPL (1429.4 local). Prime regularises zeta at 10%.
        q_prime = max(1, half // 10)
        q_zeta  = half - q_prime
        p  = torch.tensor(get_primes(q_prime), dtype=torch.float32)
        pf = (1.0 / (p ** 0.75))
        zf = get_zeta_freqs(q_zeta)
        pf = pf / pf.max()
        zf = zf / zf.max()
        pf_padded = pf.repeat(q_zeta // q_prime + 1)[:q_zeta]
        freqs = torch.stack([zf, pf_padded], dim=1).flatten()[:half]
        return {"type": "additive", "learnable": False, "freqs": freqs}
    elif pe_name == "hybrid_50z":
        # v1.4.0: 50/50 interleaved — plateau at -0.3% (comparison point)
        q = half // 2
        p  = torch.tensor(get_primes(q), dtype=torch.float32)
        pf = 1.0 / (p ** 0.75)
        zf = get_zeta_freqs(q)
        pf = pf[pf.argsort(descending=True)] / pf.max()
        zf = zf[zf.argsort(descending=True)] / zf.max()
        freqs = torch.stack([pf, zf], dim=1).flatten()
        return {"type": "additive", "learnable": False, "freqs": freqs}
    elif pe_name == "random_irr":
        # Fractional parts of sqrt(prime) — provably irrational, zero number-theoretic structure
        p = torch.tensor(get_primes(half), dtype=torch.float32)
        freqs = p.sqrt().frac()
        freqs = freqs / freqs.max()
        return {"type": "additive", "learnable": False, "freqs": freqs}
    elif pe_name == "learned":
        freqs = 1.0 / (10000 ** (torch.arange(half).float() / half))
        return {"type": "additive", "learnable": True, "freqs": freqs}
    elif pe_name == "zeta_rope":
        # RoPE architecture with zeta-zero freqs — apples-to-apples vs rope
        rope_half = half // 2
        zf  = get_zeta_freqs(rope_half)
        geo = 1.0 / (10000 ** (torch.arange(rope_half).float() / rope_half))
        zf  = zf * (geo.mean() / zf.mean())  # match magnitude for gradient stability
        return {"type": "rope", "learnable": False, "freqs": zf}
    elif pe_name == "random_irr_rope":
        # CONTROL: RoPE arch with random irrational freqs
        # Isolates: does zeta_rope win due to architecture or frequency structure?
        rope_half = half // 2
        p = torch.tensor(get_primes(rope_half), dtype=torch.float32)
        freqs = p.sqrt().frac()
        freqs = freqs / freqs.max()
        geo = 1.0 / (10000 ** (torch.arange(rope_half).float() / rope_half))
        freqs = freqs * (geo.mean() / freqs.mean())
        return {"type": "rope", "learnable": False, "freqs": freqs}
    else:
        raise ValueError(f"Unknown PE: {pe_name}")

print("Frequency specs ready.")
for pe in CFG["pe_variants"]:
    spec = freq_spec(pe, CFG["d_model"])
    fmin = spec["freqs"].min().item() if spec["freqs"] is not None else None
    fmax = spec["freqs"].max().item() if spec["freqs"] is not None else None
    print(f"  {pe:<14} type={spec['type']:<10} learnable={spec['learnable']} "
          f"freq_range=[{fmin:.4f}, {fmax:.4f}]" if fmin is not None else f"  {pe}")

In [ ]:
# ─── Cell 5: Model — FlashAttention-2 + SDPA fallback ─────────────────────────
# Version: v0.3.1-h100

class AdditivePE(nn.Module):
    """Additive sinusoidal-style PE with arbitrary frequency set."""
    def __init__(self, d_model: int, freqs: torch.Tensor, max_len: int = 65536,
                 learnable: bool = False):
        super().__init__()
        self.d_model = d_model
        half = d_model // 2
        freqs = freqs[:half]
        if learnable:
            self.freqs = nn.Parameter(freqs.clone())
        else:
            self.register_buffer("freqs", freqs)
        self.learnable = learnable
        # Precompute for max_len if not learnable
        if not learnable:
            pos = torch.arange(max_len).float().unsqueeze(1)       # (L, 1)
            phases = pos * freqs.unsqueeze(0)                       # (L, half)
            pe = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)  # (L, d)
            self.register_buffer("_pe_cache", pe)
        else:
            self._pe_cache = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        if self.learnable or self._pe_cache is None:
            pos = torch.arange(T, device=x.device).float().unsqueeze(1)
            phases = pos * self.freqs.unsqueeze(0)
            pe = torch.cat([torch.sin(phases), torch.cos(phases)], dim=-1)
        else:
            pe = self._pe_cache[:T]
        return x + pe.unsqueeze(0).to(x.dtype)


class Attention(nn.Module):
    """Multi-head attention: FlashAttn-2 if available, else PyTorch SDPA."""
    def __init__(self, d_model: int, n_heads: int, dropout: float,
                 attn_type: str = "standard",
                 rope_freqs: Optional[torch.Tensor] = None,
                 alibi_slopes: Optional[torch.Tensor] = None):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.scale    = self.head_dim ** -0.5
        self.attn_type = attn_type
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        if rope_freqs is not None:
            self.register_buffer("rope_freqs", rope_freqs)
        if alibi_slopes is not None:
            self.register_buffer("alibi_slopes", alibi_slopes)

    def _apply_rope(self, x: torch.Tensor) -> torch.Tensor:
        B, H, T, D = x.shape
        t = torch.arange(T, device=x.device).float()
        freqs = torch.outer(t, self.rope_freqs[:D//2])   # (T, D//2)
        cos = freqs.cos()[None, None].to(x.dtype)
        sin = freqs.sin()[None, None].to(x.dtype)
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1*cos - x2*sin, x1*sin + x2*cos], dim=-1).flatten(-2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)

        # PyTorch SDPA — auto-selects FlashAttention-2 on H100+bf16
        q, k, v = qkv.permute(2,0,3,1,4).unbind(0)         # each (B,H,T,D)
        if self.attn_type == "rope":
            q, k = self._apply_rope(q), self._apply_rope(k)
        if self.attn_type == "alibi":
                # Compute ALiBi bias
                pos = torch.arange(T, device=x.device)
                dist = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs().float()  # (T,T)
                bias = -self.alibi_slopes.view(-1,1,1) * dist.unsqueeze(0)  # (H,T,T)
                # Causal mask
                causal = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
                attn_bias = bias + causal.unsqueeze(0)
            out = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_bias, dropout_p=0.0)
        else:
            out = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, is_causal=True, dropout_p=0.0)
        out = out.transpose(1,2).reshape(B, T, C)

        return self.proj(self.drop(out))


class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float,
                 attn_type: str, rope_freqs=None, alibi_slopes=None):
        super().__init__()
        self.attn  = Attention(d_model, n_heads, dropout, attn_type, rope_freqs, alibi_slopes)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class PrimePEModel(nn.Module):
    """
    Decoder-only transformer with swappable PE.
    Version: v0.3.1-h100 | Author: Knack
    Change Log:
        v0.3.0  — Initial Phase 3 model
        v0.3.1  — H100: FlashAttn-2, bf16, SDPA fallback, pre-cached PE
    """
    def __init__(self, cfg: dict, pe_name: str):
        super().__init__()
        self.pe_name = pe_name
        d = cfg["d_model"]
        spec = freq_spec(pe_name, d)

        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.embed_drop = nn.Dropout(cfg["dropout"])

        # Additive PE (sinusoidal, prime_*, zeta, hybrid, random_irr, learned)
        self.pe = None
        if spec["type"] == "additive":
            self.pe = AdditivePE(d, spec["freqs"], learnable=spec["learnable"])

        # Build attention kwargs
        attn_type = spec["type"]  # "additive", "rope", "alibi"
        rope_freqs = spec["freqs"] if attn_type == "rope" else None
        if attn_type == "alibi":
            n = cfg["n_heads"]
            slopes = torch.tensor([2**(-8*i/n) for i in range(1, n+1)])
            alibi_slopes = slopes
        else:
            alibi_slopes = None

        self.blocks = nn.ModuleList([
            Block(d, cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"],
                  attn_type if attn_type != "additive" else "standard",
                  rope_freqs, alibi_slopes)
            for _ in range(cfg["n_layers"])
        ])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.head.weight = self.embed.weight
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(ids) * math.sqrt(self.cfg["d_model"])  # standard embedding scale
        if self.pe:
            x = self.pe(x)
        x = self.embed_drop(x)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Sanity check
torch.manual_seed(0)
_m = PrimePEModel(CFG, "zeta").to(DEVICE)
print(f"Model params: {_m.n_params()/1e6:.1f}M")
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=DTYPE):
    _o = _m(torch.randint(0, 1000, (2, 128)).to(DEVICE))
print(f"Forward pass OK — output shape: {_o.shape} ✅")
del _m, _o; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ─── Cell 6: Data loading ──────────────────────────────────────────────────────

from datasets import load_dataset
from transformers import GPT2TokenizerFast
from torch.utils.data import Dataset, DataLoader

print("Loading WikiText-103...")
raw = load_dataset("wikitext", "wikitext-103-raw-v1")
tok = GPT2TokenizerFast.from_pretrained("gpt2")
tok.pad_token = tok.eos_token

def _tokenise(split: str, seq_len: int) -> torch.Tensor:
    text = "\n".join(raw[split]["text"])
    ids  = torch.tensor(tok.encode(text), dtype=torch.long)
    n    = (len(ids) // seq_len) * seq_len
    return ids[:n].reshape(-1, seq_len)

class ChunkDataset(Dataset):
    def __init__(self, chunks): self.c = chunks
    def __len__(self): return len(self.c)
    def __getitem__(self, i): return self.c[i][:-1], self.c[i][1:]

SL = CFG["train_seq_len"] + 1
print(f"Tokenising (seq_len={SL-1})...")
train_ds = ChunkDataset(_tokenise("train",      SL))
val_ds   = ChunkDataset(_tokenise("validation", SL))

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} sequences")

In [ ]:
# ─── Cell 7: Training loop ─────────────────────────────────────────────────────
# Version: v0.3.1-h100 — bf16 AMP, gradient accumulation, cosine LR

from itertools import cycle

def cosine_lr(step: int, cfg: dict) -> float:
    if step < cfg["warmup_steps"]:
        return cfg["lr"] * step / max(1, cfg["warmup_steps"])
    t = (step - cfg["warmup_steps"]) / max(1, cfg["max_steps"] - cfg["warmup_steps"])
    return cfg["lr"] * 0.5 * (1 + math.cos(math.pi * t))

@torch.no_grad()
def evaluate(model, loader, max_batches=100) -> float:
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            logits = model(x)
        loss = nn.functional.cross_entropy(logits.float().reshape(-1, CFG["vocab_size"]), y.reshape(-1))
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

def train_variant(pe_name: str, cfg: dict) -> dict:
    print(f"\n{'═'*60}")
    print(f"  PE: {pe_name}")
    print(f"{'═'*60}")

    torch.manual_seed(cfg["seed"])
    model = PrimePEModel(cfg, pe_name).to(DEVICE)

    if cfg["use_compile"]:
        print("  Compiling model...")
        model = torch.compile(model, mode="default")  # reduce-overhead uses CUDAGraphs which crash on training loops

    raw_m = model._orig_mod if hasattr(model, "_orig_mod") else model
    print(f"  Params: {raw_m.n_params()/1e6:.1f}M")

    opt = torch.optim.AdamW(
        model.parameters(), lr=cfg["lr"],
        weight_decay=cfg["weight_decay"], betas=(0.9, 0.95), fused=True
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))

    results = {
        "pe": pe_name, "steps": [], "train_loss": [], "val_loss": [],
        "train_steps_dense": [], "train_loss_dense": [],  # every grad step
        "final_val_loss": None, "final_val_ppl": None, "time_s": None,
    }

    it = cycle(train_loader)
    t0 = time.time()
    step = 0
    running_loss = 0.0
    opt.zero_grad()

    while step < cfg["max_steps"]:
        lr = cosine_lr(step, cfg)
        for g in opt.param_groups: g["lr"] = lr

        x, y = next(it)
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=DTYPE):
            logits = model(x)
            loss = nn.functional.cross_entropy(
                logits.float().reshape(-1, cfg["vocab_size"]),
                y.reshape(-1)
            ) / cfg["grad_accum"]

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if (step + 1) % cfg["grad_accum"] == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip_grad"])
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            # Dense loss tracking every grad step
            results["train_steps_dense"].append(step)
            results["train_loss_dense"].append(loss.item() * cfg["grad_accum"])

            if step % cfg["eval_every"] == 0 or step == cfg["max_steps"] - 1:
                val = evaluate(model, val_loader)
                elapsed = time.time() - t0
                tokens_per_s = (step * cfg["batch_size"] * cfg["train_seq_len"]) / elapsed
                print(f"  step {step:6d} | train {running_loss:.4f} | val {val:.4f} "
                      f"| ppl {math.exp(val):.1f} | {tokens_per_s/1e3:.1f}K tok/s | {elapsed:.0f}s")
                results["steps"].append(step)
                results["train_loss"].append(running_loss)
                results["val_loss"].append(val)
                running_loss = 0.0

        step += 1

    final = evaluate(model, val_loader, max_batches=500)
    results["final_val_loss"] = final
    results["final_val_ppl"]  = math.exp(final)
    results["time_s"]         = time.time() - t0
    print(f"  ✅ Final — val_loss: {final:.4f}  PPL: {math.exp(final):.1f}  "
          f"time: {results['time_s']/60:.1f}min")

    # Save weights (unwrap compiled model)
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    path = f"{cfg['results_dir']}/ckpt_{pe_name}.pt"
    torch.save(raw_model.state_dict(), path)
    results["ckpt"] = path

    del model; gc.collect(); torch.cuda.empty_cache()
    return results

print("Training loop ready.")

In [ ]:
# ─── Cell 8: Run all variants ──────────────────────────────────────────────────

all_results = {}
for pe in CFG["pe_variants"]:
    r = train_variant(pe, CFG)
    all_results[pe] = r
    with open(f"{CFG['results_dir']}/training_results.json", "w") as f:
        json.dump(all_results, f, indent=2, default=str)
    print(f"  Checkpoint saved after {pe}")

print("\n✅ All training complete.")

In [ ]:
# ─── Cell 9: Long-context perplexity ──────────────────────────────────────────
# Key test: does PPL degrade more slowly for ZetaPE as context grows?

@torch.no_grad()
def ppl_at_len(model, seq_len: int, n_batches: int = 50) -> float:
    chunks = _tokenise("test", seq_len + 1)
    ds     = ChunkDataset(chunks)
    loader = DataLoader(ds, batch_size=2, shuffle=False, drop_last=True)
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= n_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        try:
            with torch.autocast(device_type="cuda", dtype=DTYPE):
                logits = model(x)
            loss = nn.functional.cross_entropy(
                logits.float().reshape(-1, CFG["vocab_size"]), y.reshape(-1))
            losses.append(loss.item())
        except RuntimeError as e:
            print(f"    OOM at len={seq_len}: skipping")
            break
    return math.exp(np.mean(losses)) if losses else float("nan")

print("Long-context PPL evaluation...")
ppl_results = {}

for pe in CFG["pe_variants"]:
    ckpt = all_results[pe].get("ckpt")
    if not ckpt: continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    ppl_results[pe] = {}
    row = f"  {pe:<14}"
    for sl in CFG["eval_seq_lens"]:
        ppl = ppl_at_len(model, sl)
        ppl_results[pe][sl] = ppl
        row += f" | {sl//1024}K→{ppl:.1f}"
    print(row)

    del model; gc.collect(); torch.cuda.empty_cache()

with open(f"{CFG['results_dir']}/ppl_results.json", "w") as f:
    json.dump(ppl_results, f, indent=2)
print("\n✅ PPL eval complete.")

In [ ]:
# ─── Cell 10: Lost-in-the-Middle benchmark ────────────────────────────────────
# THE key number: retrieval accuracy at position 0.5 (exact middle)

@torch.no_grad()
def litm_benchmark(model, cfg: dict) -> dict:
    """
    Synthetic needle retrieval across context positions.
    Needle = a specific rare token placed at position frac * ctx_len.
    Query  = does the model predict the needle at that position?
    This directly measures the U-shaped attention degradation.
    """
    model.eval()
    ctx  = cfg["litm_context_len"]
    fracs = cfg["litm_positions"]
    results = {}

    for frac in fracs:
        needle_pos = max(1, int(frac * ctx) - 1)
        correct = 0

        # Fixed haystack and needle tokens — removes vocabulary frequency artifacts
        # Both tokens are from common vocabulary so model knows them well
        HAYSTACK_TOK = 318   # " is" — very common, stable representation
        NEEDLE_TOK   = 7400  # " Constantinople" — distinctive, uncommon
        for _ in range(cfg["litm_samples"]):
            ids = torch.full((ctx,), HAYSTACK_TOK, dtype=torch.long)
            ids[needle_pos] = NEEDLE_TOK
            needle = NEEDLE_TOK

            inp = ids.unsqueeze(0).to(DEVICE)
            try:
                with torch.autocast(device_type="cuda", dtype=DTYPE):
                    logits = model(inp)   # (1, ctx, vocab)
                # Position needle_pos-1 should predict needle
                pred = logits[0, needle_pos - 1].argmax().item()
                if pred == needle:
                    correct += 1
            except RuntimeError:
                break

        acc = correct / cfg["litm_samples"]
        results[frac] = acc

    return results

print("Running Lost-in-the-Middle benchmark...")
print(f"Context: {CFG['litm_context_len']} tokens | Samples: {CFG['litm_samples']} per position")
litm_results = {}

for pe in CFG["pe_variants"]:
    ckpt = all_results[pe].get("ckpt")
    if not ckpt: continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    litm = litm_benchmark(model, CFG)
    litm_results[pe] = litm

    mid = litm.get(0.5, 0)
    vals = " | ".join(f"{int(k*100)}%:{v:.1%}" for k, v in litm.items())
    flag = " ← KEY" if mid > 0 else ""
    print(f"  {pe:<14} {vals}{flag}")

    del model; gc.collect(); torch.cuda.empty_cache()

with open(f"{CFG['results_dir']}/litm_results.json", "w") as f:
    json.dump(litm_results, f, indent=2)
print("\n✅ LitM benchmark complete.")

In [ ]:
# ─── Cell 11: Attention Distance Analysis ─────────────────────────────────────
# Mechanistic test: do zeta-frequency dimensions attend at longer ranges
# than prime-frequency dimensions?
#
# If yes: the 90/10 split reflects a genuine structural decomposition —
# zeta = long-range structural (paragraph/topic), prime = local syntactic.
# This turns the empirical 90/10 finding into a mechanistic claim.
#
# Method: hook attention weights, compute mean attended distance per head,
# correlate with each head's dominant frequency band (zeta vs prime).
# Version: v0.3.2-h100 | Author: Knack

import numpy as np
from collections import defaultdict

@torch.no_grad()
def attention_distance_analysis(model, cfg: dict, n_batches: int = 100) -> dict:
    """
    For each attention head in each layer, compute the mean distance between
    query position and the attended key position (weighted by attention weight).

    Returns:
        {
          'per_head': {layer: {head: mean_distance}},
          'by_freq_type': {'zeta': mean_dist, 'prime': mean_dist, 'other': mean_dist},
          'freq_band_distances': [(freq_value, mean_distance), ...],
        }
    """
    model.eval()
    hooks = []
    attn_weights_store = defaultdict(list)  # {(layer, head): [mean_dist, ...]}

    # ── Register hooks on every attention block ────────────────────────────────
    def make_hook(layer_idx):
        def hook(module, input, output):
            # Re-compute attention weights from Q, K for distance tracking
            # We hook the Attention module's forward; x is input[0]
            x = input[0]
            B, T, C = x.shape
            with torch.no_grad():
                qkv = module.qkv(x).reshape(B, T, 3, module.n_heads, module.head_dim)
                q, k, _ = qkv.permute(2,0,3,1,4).unbind(0)  # (B,H,T,D)
                # Raw attention scores (no masking for distance analysis)
                scores = (q @ k.transpose(-2,-1)) * module.scale  # (B,H,T,T)
                # Causal softmax
                mask = torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool), diagonal=1)
                scores = scores.masked_fill(mask, float('-inf'))
                weights = scores.softmax(-1)  # (B,H,T,T)
                # Mean attended distance for each head
                pos = torch.arange(T, device=x.device).float()
                # distance[i,j] = |i - j|
                dist_mat = (pos.unsqueeze(0) - pos.unsqueeze(1)).abs()  # (T,T)
                # weighted mean distance per head: sum over j of weight[i,j]*dist[i,j], mean over i
                mean_dist = (weights * dist_mat.unsqueeze(0).unsqueeze(0)).sum(-1).mean(-1)  # (B,H)
                mean_dist = mean_dist.mean(0)  # (H,) — average over batch
                for h in range(module.n_heads):
                    attn_weights_store[(layer_idx, h)].append(mean_dist[h].item())
        return hook

    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    for layer_idx, block in enumerate(raw_model.blocks):
        h = block.attn.register_forward_hook(make_hook(layer_idx))
        hooks.append(h)

    # ── Run forward passes ────────────────────────────────────────────────────
    chunks = _tokenise("test", cfg["train_seq_len"] + 1)
    ds = ChunkDataset(chunks)
    loader = DataLoader(ds, batch_size=4, shuffle=False, drop_last=True)
    for i, (x, _) in enumerate(loader):
        if i >= n_batches: break
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            model(x.to(DEVICE))

    for h in hooks: h.remove()

    # ── Aggregate per-head distances ──────────────────────────────────────────
    per_head = {}
    for (layer, head), dists in attn_weights_store.items():
        per_head[(layer, head)] = float(np.mean(dists))

    return {"per_head": per_head}


def classify_pe_freqs(pe_name: str, cfg: dict) -> list:
    """
    Returns a label for each of the n_heads heads: 'zeta', 'prime', or 'other'.
    For additive PE: the first d//2 dims are sin, second are cos.
    Each head attends to head_dim dimensions; we label by which freq band dominates.
    Simplified: for hybrid_90z, first 90% of freq dims are zeta, last 10% prime.
    """
    d = cfg["d_model"]
    n_heads = cfg["n_heads"]
    head_dim = d // n_heads
    half = d // 2

    # Build freq type per dimension
    dim_type = []
    if pe_name == "zeta":
        dim_type = ["zeta"] * d
    elif pe_name == "hybrid_90z":
        q_prime = max(1, half // 10)
        q_zeta  = half - q_prime
        # Interleaved: [z0,p0, z1,p1, ...] with zeta dominant
        for i in range(half):
            dim_type.append("prime" if i % (half // q_prime) == 1 else "zeta")
        dim_type = dim_type * 2  # sin + cos halves
    elif pe_name == "hybrid_50z":
        for i in range(half):
            dim_type.append("zeta" if i % 2 == 0 else "prime")
        dim_type = dim_type * 2
    elif pe_name == "prime_05":
        dim_type = ["prime"] * d
    elif pe_name in ("sinusoidal", "rope", "random_irr", "learned", "alibi"):
        dim_type = ["other"] * d
    else:
        dim_type = ["other"] * d

    # Label each head by majority freq type in its slice
    head_labels = []
    for h in range(n_heads):
        start = h * head_dim
        end   = start + head_dim
        slice_types = dim_type[start:end]
        counts = {t: slice_types.count(t) for t in set(slice_types)}
        head_labels.append(max(counts, key=counts.get))
    return head_labels


print("Running attention distance analysis...")
print(f"n_batches=100, seq_len={CFG['train_seq_len']}")
dist_results = {}

# Run on the key variants
for pe in ["zeta", "hybrid_90z", "hybrid_50z", "sinusoidal", "random_irr"]:
    ckpt = all_results.get(pe, {}).get("ckpt")
    if not ckpt or not os.path.exists(ckpt):
        print(f"  {pe}: no checkpoint, skipping")
        continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

    result = attention_distance_analysis(model, CFG)
    head_labels = classify_pe_freqs(pe, CFG)

    # Group mean distances by freq type
    by_type = defaultdict(list)
    n_layers = CFG["n_layers"]
    n_heads  = CFG["n_heads"]
    for layer in range(n_layers):
        for head in range(n_heads):
            d = result["per_head"].get((layer, head), float("nan"))
            label = head_labels[head]
            by_type[label].append(d)

    summary = {t: float(np.nanmean(v)) for t, v in by_type.items()}
    dist_results[pe] = {"per_head": result["per_head"], "by_type": summary}

    print(f"  {pe:<14} ", end="")
    for t, v in sorted(summary.items()):
        print(f"{t}={v:.1f}tok ", end="")
    print()

    del model; gc.collect(); torch.cuda.empty_cache()

# ── Key verdict ───────────────────────────────────────────────────────────────
print("\n" + "─"*60)
print("MECHANISTIC TEST: zeta dims attend further than prime dims?")
h90 = dist_results.get("hybrid_90z", {}).get("by_type", {})
if "zeta" in h90 and "prime" in h90:
    delta = h90["zeta"] - h90["prime"]
    verdict = "YES ✅" if delta > 2 else "MARGINAL" if delta > 0 else "NO"
    print(f"  hybrid_90z: zeta heads={h90['zeta']:.1f}tok, prime heads={h90['prime']:.1f}tok, Δ={delta:+.1f}")
    print(f"  Verdict: {verdict}")
    if delta > 2:
        print("  → Decomposition is structural, not coincidental.")
        print("  → The music analogy is empirically grounded.")

with open(f"{CFG['results_dir']}/attn_distance.json", "w") as f:
    json.dump({k: {"by_type": v["by_type"]} for k,v in dist_results.items()}, f, indent=2)
print("\n✅ Attention distance analysis complete.")


In [ ]:
# ─── Cell 12: Position Shuffle Experiment ─────────────────────────────────────
# Tests whether zeta PE encodes positions more distinctly than baselines.
#
# Method: run model on a normal sequence, record output logits.
# Then swap two position encoding vectors (not the tokens) and re-run.
# Measure KL divergence between original and shuffled outputs.
#
# Prediction: higher KL = model more sensitive to correct positional address.
# If zeta PE encodes positions more distinctly, shuffling should hurt more.
# This is a clean mechanistic test that doesn't require long contexts.
# Version: v0.3.3 | Author: Knack

@torch.no_grad()
def position_shuffle_test(model, cfg: dict, n_samples: int = 200,
                          swap_distance: int = 50) -> dict:
    """
    For each sample:
      1. Run model on sequence, get output logits
      2. Swap PE vectors at positions i and i+swap_distance
         (patch the AdditivePE cache directly)
      3. Re-run, measure KL(original || shuffled)
    Higher mean KL = PE encodes positions more distinctly.
    """
    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model

    # Only works for additive PE — skip RoPE/ALiBi variants
    if raw_model.pe is None:
        return {"mean_kl": float("nan"), "note": "rotary/alibi — no additive PE to swap"}

    T    = cfg["train_seq_len"]
    kls  = []

    for _ in range(n_samples):
        ids = torch.randint(100, 5000, (1, T)).to(cfg["device"])
        # Swap position i and i+swap_distance in the PE cache
        swap_i = torch.randint(10, T - swap_distance - 10, (1,)).item()
        swap_j = swap_i + swap_distance

        # Original output
        with torch.autocast(device_type="cuda", dtype=cfg["dtype"]):
            logits_orig = model(ids).float()  # (1, T, vocab)

        # Swap PE vectors in the cache
        if raw_model.pe._cache is not None:
            pe_cache = raw_model.pe._cache  # (max_len, d)
            orig_i = pe_cache[swap_i].clone()
            orig_j = pe_cache[swap_j].clone()
            pe_cache[swap_i] = orig_j
            pe_cache[swap_j] = orig_i

            # Shuffled output
            with torch.autocast(device_type="cuda", dtype=cfg["dtype"]):
                logits_swap = model(ids).float()

            # Restore
            pe_cache[swap_i] = orig_i
            pe_cache[swap_j] = orig_j

            # KL at swapped positions only (where the change should matter most)
            p = logits_orig[0, swap_i].softmax(-1).clamp(min=1e-9)
            q = logits_swap[0, swap_i].softmax(-1).clamp(min=1e-9)
            kl = (p * (p / q).log()).sum().item()
            kls.append(kl)

    if not kls:
        return {"mean_kl": float("nan"), "note": "no learnable cache"}

    return {
        "mean_kl":   float(np.mean(kls)),
        "median_kl": float(np.median(kls)),
        "swap_dist": swap_distance,
        "n_samples": len(kls),
    }


print("Running position shuffle experiment...")
print(f"Swap distance: 50 tokens | Samples: 200 per variant")
print(f"Prediction: zeta > sinusoidal > random_irr (higher KL = more distinct encoding)\n")

shuffle_results = {}
# Run on additive PE variants only (RoPE/ALiBi have no swappable cache)
additive_pes = [p for p in CFG["pe_variants"]
                if p not in ("rope", "zeta_rope", "random_irr_rope", "alibi")]

for pe in additive_pes:
    ckpt = all_results.get(pe, {}).get("ckpt")
    if not ckpt or not os.path.exists(ckpt):
        print(f"  {pe}: no checkpoint")
        continue
    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    res = position_shuffle_test(model, CFG)
    shuffle_results[pe] = res

    if not math.isnan(res["mean_kl"]):
        print(f"  {pe:<18} mean_KL={res['mean_kl']:.4f}  median_KL={res['median_kl']:.4f}")
    else:
        print(f"  {pe:<18} {res.get('note', 'no result')}")

    del model; gc.collect(); torch.cuda.empty_cache()

# Verdict
print("\n" + "─"*60)
zeta_kl = shuffle_results.get("zeta", {}).get("mean_kl", float("nan"))
rand_kl  = shuffle_results.get("random_irr", {}).get("mean_kl", float("nan"))
sin_kl   = shuffle_results.get("sinusoidal", {}).get("mean_kl", float("nan"))
print(f"zeta KL:        {zeta_kl:.4f}")
print(f"random_irr KL:  {rand_kl:.4f}")
print(f"sinusoidal KL:  {sin_kl:.4f}")
if not any(math.isnan(x) for x in [zeta_kl, rand_kl, sin_kl]):
    if zeta_kl > rand_kl and zeta_kl > sin_kl:
        print("→ Zeta PE encodes positions most distinctly ✅")
        print("→ Model is most sensitive to positional address corruption under zeta PE")
    elif zeta_kl > sin_kl:
        print("→ Zeta beats sinusoidal but not random_irr — structure not sole factor")
    else:
        print("→ No clear advantage — positional distinctness similar across variants")

with open(f"{CFG['results_dir']}/shuffle_results.json", "w") as f:
    json.dump(shuffle_results, f, indent=2)
print("\n✅ Position shuffle experiment complete.")


In [ ]:
# ─── Cell 13: Prime Resonance Probe ───────────────────────────────────────────
# Tests the Arithmetic Relational Position Hypothesis (v1.6.0):
# If prime frequencies create resonance peaks at prime-multiple distances,
# attention weight vs distance should show a 'comb' pattern at p, 2p, 3p...
# for each prime p in the frequency set.
#
# This is categorically different from sinusoidal (smooth decay) and
# random_irr (no prime structure). If confirmed, it demonstrates the model
# has learned to use prime periods as a POSITIONAL GRAMMAR — encoding
# arithmetic relationships between positions, not just distances.
#
# Prime Resonance Score (PRS): ratio of mean attention at prime-multiple
# distances vs non-prime-multiple distances. PRS > 1.0 = resonance present.
# Version: v0.3.4 | Author: Knack

PRIME_SET = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]  # primes to probe

@torch.no_grad()
def prime_resonance_probe(model, cfg: dict, n_batches: int = 150) -> dict:
    """
    Computes attention weight as a function of distance, then measures
    whether attention is elevated at prime-multiple distances (PRS > 1.0).

    Returns:
        attn_by_dist: {distance: mean_attention_weight}
        prs_by_prime: {prime: PRS score}
        prs_total: mean PRS across all primes
    """
    model.eval()
    T = cfg["train_seq_len"]

    # Accumulate attention weights by distance
    dist_attn_sum   = torch.zeros(T, device=DEVICE)
    dist_attn_count = torch.zeros(T, device=DEVICE)
    hooks = []

    def make_hook(layer_idx):
        def hook(module, inp, out):
            x = inp[0]; B, Tl, C = x.shape
            with torch.no_grad():
                qkv = module.qkv(x).reshape(B, Tl, 3, module.n_heads, module.head_dim)
                q, k, _ = qkv.permute(2,0,3,1,4).unbind(0)
                scores   = (q @ k.transpose(-2,-1)) * module.scale
                mask     = torch.triu(torch.ones(Tl,Tl,device=x.device,dtype=torch.bool), diagonal=1)
                weights  = scores.masked_fill(mask, float("-inf")).softmax(-1)
                # weights: (B, H, T, T) — average over batch and heads
                w_mean = weights.mean(dim=(0,1))  # (T, T)
                for delta in range(1, Tl):
                    # Diagonal at offset delta: positions where key is delta behind query
                    diag_vals = w_mean.diagonal(offset=-delta)  # (T-delta,)
                    dist_attn_sum[delta]   += diag_vals.sum()
                    dist_attn_count[delta] += diag_vals.numel()
        return hook

    raw_model = model._orig_mod if hasattr(model, '_orig_mod') else model
    for li, block in enumerate(raw_model.blocks):
        hooks.append(block.attn.register_forward_hook(make_hook(li)))

    chunks = _tokenise("test", T + 1)
    ds     = ChunkDataset(chunks)
    loader = DataLoader(ds, batch_size=2, shuffle=False, drop_last=True)
    for i, (x, _) in enumerate(loader):
        if i >= n_batches: break
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            model(x.to(DEVICE))

    for h in hooks: h.remove()

    # Normalise to get mean attention at each distance
    mask = dist_attn_count > 0
    attn_by_dist = torch.zeros(T)
    attn_by_dist[mask] = (dist_attn_sum[mask] / dist_attn_count[mask]).cpu()

    # Compute PRS for each prime
    distances = torch.arange(1, T)
    prs_by_prime = {}

    for p in PRIME_SET:
        prime_multiples = torch.zeros(T-1, dtype=torch.bool)
        for k in range(1, T // p + 1):
            if k * p < T:
                prime_multiples[k * p - 1] = True

        non_multiples = ~prime_multiples
        if prime_multiples.sum() == 0 or non_multiples.sum() == 0:
            prs_by_prime[p] = float("nan")
            continue

        mean_at_multiples    = attn_by_dist[1:][prime_multiples].mean().item()
        mean_at_non_multiples = attn_by_dist[1:][non_multiples].mean().item()

        prs_by_prime[p] = mean_at_multiples / mean_at_non_multiples \
                           if mean_at_non_multiples > 0 else float("nan")

    valid_prs = [v for v in prs_by_prime.values() if not (v != v)]
    prs_total = float(sum(valid_prs) / len(valid_prs)) if valid_prs else float("nan")

    return {
        "attn_by_dist": attn_by_dist.tolist(),
        "prs_by_prime": prs_by_prime,
        "prs_total":    prs_total,
    }


print("Running Prime Resonance Probe...")
print(f"Primes tested: {PRIME_SET}")
print(f"Prediction: prime_05 PRS > 1.2 | sinusoidal PRS ≈ 1.0 | random_irr PRS ≈ 1.0\n")

resonance_results = {}

for pe in CFG["pe_variants"]:
    ckpt = all_results.get(pe, {}).get("ckpt")
    if not ckpt or not os.path.exists(ckpt):
        print(f"  {pe}: no checkpoint")
        continue

    model = PrimePEModel(CFG, pe).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    result = prime_resonance_probe(model, CFG)
    resonance_results[pe] = result

    prs  = result["prs_total"]
    pprs = " ".join(f"p{p}:{v:.3f}" for p,v in list(result["prs_by_prime"].items())[:5])
    flag = " ← RESONANCE ✅" if prs > 1.05 else " ← flat" if prs < 1.02 else ""
    print(f"  {pe:<18} PRS={prs:.4f}  [{pprs}]{flag}")

    del model; gc.collect(); torch.cuda.empty_cache()

# ── Verdict ───────────────────────────────────────────────────────────────────
print("\n" + "─"*60)
print("PRIME RESONANCE VERDICT")
prime_prs  = resonance_results.get("prime_05", {}).get("prs_total", float("nan"))
zeta_prs   = resonance_results.get("zeta",     {}).get("prs_total", float("nan"))
rand_prs   = resonance_results.get("random_irr",{}).get("prs_total", float("nan"))
sin_prs    = resonance_results.get("sinusoidal",{}).get("prs_total", float("nan"))
h90_prs    = resonance_results.get("hybrid_90z",{}).get("prs_total", float("nan"))

print(f"  prime_05:   PRS={prime_prs:.4f}")
print(f"  hybrid_90z: PRS={h90_prs:.4f}")
print(f"  zeta:       PRS={zeta_prs:.4f}")
print(f"  sinusoidal: PRS={sin_prs:.4f}")
print(f"  random_irr: PRS={rand_prs:.4f}")

if not any(v!=v for v in [prime_prs, rand_prs, sin_prs]):
    if prime_prs > rand_prs + 0.02 and prime_prs > sin_prs + 0.02:
        print("\n  ✅ ARITHMETIC RELATIONAL POSITION HYPOTHESIS SUPPORTED")
        print("  Prime frequencies create resonance peaks at prime-multiple distances.")
        print("  Model has learned to use prime periods as a POSITIONAL GRAMMAR.")
        print("  This is categorically different from sinusoidal/random PE.")
    elif prime_prs > rand_prs:
        print("\n  🔬 MARGINAL: Resonance present but weak — may need more training")
    else:
        print("\n  ❌ NO RESONANCE at this training scale — hypothesis not confirmed yet")
        print("  Note: 20K steps may be insufficient. Spectral curriculum may be needed.")

with open(f"{CFG['results_dir']}/resonance_results.json", "w") as f:
    safe = {k: {kk: vv if not isinstance(vv, list) else vv[:200]
                for kk,vv in v.items()} for k,v in resonance_results.items()}
    json.dump(safe, f, indent=2)
print("\n✅ Prime Resonance Probe complete.")


In [ ]:
# ─── Cell 14: Plots ────────────────────────────────────────────────────────────

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DARK, PANEL = "#0a0a0f", "#0d1117"
PE_COL = {
    "sinusoidal":  "#90a4ae",
    "rope":        "#546e7a",
    "zeta":        "#00e5ff",
    "hybrid_90z":  "#ff4081",
    "hybrid_50z":  "#f48fb1",
    "prime_05":    "#b2ff59",
    "random_irr":  "#ff9100",
    "learned":     "#ffd740",
    "alibi":       "#ce93d8",
}

def dark_ax(figsize=(13,6)):
    fig, ax = plt.subplots(figsize=figsize, facecolor=DARK)
    ax.set_facecolor(PANEL)
    for spine in ["bottom","left"]: ax.spines[spine].set_color("#546e7a")
    for spine in ["top","right"]:   ax.spines[spine].set_visible(False)
    ax.tick_params(colors="#90a4ae")
    return fig, ax

def lbl(ax, txt, color="#90a4ae"):
    ax.set_xlabel(txt, color=color)
def ttl(ax, txt):
    ax.set_title(txt, color="#00e5ff", fontsize=13, pad=10)
def leg(ax):
    ax.legend(facecolor=PANEL, edgecolor="#546e7a", labelcolor="#e8eaf6", fontsize=8)
def savefig(fig, name):
    path = f"{CFG['results_dir']}/{name}"
    fig.tight_layout()
    fig.savefig(path, dpi=150, facecolor=DARK)
    plt.close()
    print(f"  Saved {name}")

# ── 1. Convergence ────────────────────────────────────────────────────────────
fig, ax = dark_ax()
for pe, r in all_results.items():
    if r.get("val_loss"):
        ax.plot(r["steps"], r["val_loss"], color=PE_COL.get(pe, "white"),
                label=pe, lw=2)
lbl(ax, "Step"); ax.set_ylabel("Val Loss", color="#90a4ae")
ttl(ax, "Convergence — All PE Variants"); leg(ax)
ax.grid(True, color="#1e293b", alpha=0.6)
savefig(fig, "convergence.png")

# ── 2. PPL vs context length ──────────────────────────────────────────────────
if ppl_results:
    fig, ax = dark_ax()
    for pe, lens in ppl_results.items():
        xs = sorted(lens.keys())
        ys = [lens[x] for x in xs]
        ax.plot(xs, ys, marker="o", color=PE_COL.get(pe, "white"),
                label=pe, lw=2, ms=6)
    ax.set_xscale("log"); ax.set_yscale("log")
    lbl(ax, "Context Length (tokens)"); ax.set_ylabel("Perplexity", color="#90a4ae")
    ttl(ax, "Perplexity vs Context Length — log/log")
    leg(ax); ax.grid(True, color="#1e293b", alpha=0.6)
    ax.axvline(10000, color="#ff4081", ls="--", alpha=0.4, lw=1, label="sinusoidal alias ~10K")
    savefig(fig, "ppl_vs_context.png")

# ── 3. LitM heatmap ───────────────────────────────────────────────────────────
if litm_results:
    fracs = CFG["litm_positions"]
    pes   = list(litm_results.keys())
    mat   = [[litm_results[p].get(f, 0) for f in fracs] for p in pes]
    fig, ax = plt.subplots(figsize=(10, max(4, len(pes)*0.7+1)), facecolor=DARK)
    ax.set_facecolor(PANEL)
    im = ax.imshow(mat, aspect="auto", cmap="plasma", vmin=0, vmax=1)
    ax.set_xticks(range(len(fracs)))
    ax.set_xticklabels([f"{int(f*100)}%" for f in fracs], color="#90a4ae")
    ax.set_yticks(range(len(pes)))
    ax.set_yticklabels(pes, color="#e8eaf6")
    ax.set_xlabel("Needle position (% of context)", color="#90a4ae")
    ax.set_title(f"Lost-in-the-Middle: Retrieval Accuracy @ {CFG['litm_context_len']} tokens",
                 color="#00e5ff", fontsize=12, pad=10)
    for i, p in enumerate(pes):
        for j, f in enumerate(fracs):
            v = litm_results[p].get(f, 0)
            ax.text(j, i, f"{v:.0%}", ha="center", va="center",
                    color="white", fontsize=9, fontweight="bold")
    fig.colorbar(im, ax=ax).ax.tick_params(colors="#90a4ae")
    savefig(fig, "litm_heatmap.png")

# ── 4. LitM curves ────────────────────────────────────────────────────────────
if litm_results:
    fig, ax = dark_ax()
    for pe, res in litm_results.items():
        xs = [f*100 for f in sorted(res.keys())]
        ys = [res[f] for f in sorted(res.keys())]
        ax.plot(xs, ys, marker="o", color=PE_COL.get(pe, "white"),
                label=pe, lw=2.5, ms=8)
    ax.axvline(50, color="#ff4081", ls="--", alpha=0.5, lw=1.5)
    ax.text(51, 0.05, "middle", color="#ff4081", fontsize=8)
    ax.set_ylim(0, 1.05)
    lbl(ax, "Needle position (% of context)")
    ax.set_ylabel("Retrieval accuracy", color="#90a4ae")
    ttl(ax, "Lost-in-the-Middle — Position Accuracy")
    leg(ax); ax.grid(True, color="#1e293b", alpha=0.6)
    savefig(fig, "litm_curves.png")

# ── 5. Final PPL bar chart ────────────────────────────────────────────────────
fig, ax = dark_ax((12, 5))
pes  = list(all_results.keys())
ppls = [all_results[p].get("final_val_ppl", float("nan")) for p in pes]
bars = ax.bar(pes, ppls, color=[PE_COL.get(p, "white") for p in pes], alpha=0.85)
for bar, ppl in zip(bars, ppls):
    if not math.isnan(ppl):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{ppl:.1f}", ha="center", va="bottom", color="#e8eaf6", fontsize=8)
ax.set_ylabel("Final Val PPL (lower = better)", color="#90a4ae")
ttl(ax, "Final Validation Perplexity — All PE Variants")
ax.grid(True, axis="y", color="#1e293b", alpha=0.6)
plt.xticks(rotation=30, ha="right", color="#e8eaf6")
savefig(fig, "final_ppl_bar.png")

print("\n✅ All plots saved.")
# ── 5. Attention distance by freq type (hybrid_90z) ──────────────────────────
if dist_results:
    h90 = dist_results.get("hybrid_90z", {}).get("by_type", {})
    h50 = dist_results.get("hybrid_50z", {}).get("by_type", {})
    zeta_only = dist_results.get("zeta", {}).get("by_type", {})
    rand = dist_results.get("random_irr", {}).get("by_type", {})
    sin  = dist_results.get("sinusoidal", {}).get("by_type", {})

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)
    for ax in axes:
        ax.set_facecolor(PANEL)
        for spine in ["bottom","left"]: ax.spines[spine].set_color("#546e7a")
        for spine in ["top","right"]:   ax.spines[spine].set_visible(False)
        ax.tick_params(colors="#90a4ae")

    # Left: bar chart of mean attended distance by freq type for hybrid_90z
    ax = axes[0]
    types = list(h90.keys())
    vals  = [h90[t] for t in types]
    cols  = ["#00e5ff" if t=="zeta" else "#b2ff59" if t=="prime" else "#90a4ae" for t in types]
    bars  = ax.bar(types, vals, color=cols, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f"{v:.1f}", ha="center",
                color="#e8eaf6", fontsize=9)
    ax.set_ylabel("Mean attended distance (tokens)", color="#90a4ae")
    ax.set_title("hybrid_90z: attended distance by freq type", color="#00e5ff", fontsize=11, pad=8)
    ax.grid(True, axis="y", color="#1e293b", alpha=0.6)

    # Right: compare across PE variants (overall mean distance)
    ax = axes[1]
    pe_names, means = [], []
    for pe, res in dist_results.items():
        bt = res.get("by_type", {})
        if bt:
            pe_names.append(pe)
            means.append(float(np.mean(list(bt.values()))))
    cols2 = [PE_COL.get(p, "#90a4ae") for p in pe_names]
    bars2 = ax.bar(pe_names, means, color=cols2, alpha=0.85)
    for bar, v in zip(bars2, means):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f"{v:.1f}", ha="center",
                color="#e8eaf6", fontsize=9)
    ax.set_ylabel("Mean attended distance (tokens)", color="#90a4ae")
    ax.set_title("Overall mean attended distance by PE", color="#00e5ff", fontsize=11, pad=8)
    ax.grid(True, axis="y", color="#1e293b", alpha=0.6)
    plt.xticks(rotation=20, ha="right", color="#e8eaf6")

    fig.tight_layout()
    fig.savefig(f"{CFG['results_dir']}/attn_distance.png", dpi=150, facecolor=DARK)
    plt.close()
    print("  Saved attn_distance.png")


# ── 6. Prime Resonance — PRS bar chart ───────────────────────────────────────
if resonance_results:
    pes_r = list(resonance_results.keys())
    prs_vals = [resonance_results[p].get("prs_total", float("nan")) for p in pes_r]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)
    for ax in axes:
        ax.set_facecolor(PANEL)
        for sp in ["bottom","left"]: ax.spines[sp].set_color("#546e7a")
        for sp in ["top","right"]:   ax.spines[sp].set_visible(False)
        ax.tick_params(colors="#90a4ae")

    # Left: PRS bar chart
    ax = axes[0]
    cols = [PE_COL.get(p, "white") for p in pes_r]
    bars = ax.bar(pes_r, prs_vals, color=cols, alpha=0.85)
    ax.axhline(1.0, color="#90a4ae", ls="--", lw=1, alpha=0.6, label="baseline (no resonance)")
    for bar, v in zip(bars, prs_vals):
        if not (v != v):
            ax.text(bar.get_x()+bar.get_width()/2, v+0.002, f"{v:.3f}",
                    ha="center", color="#e8eaf6", fontsize=8)
    ax.set_ylabel("Prime Resonance Score (PRS)", color="#90a4ae")
    ax.set_title("Prime Resonance Score — All Variants\n(>1.0 = attention peaks at prime-multiple distances)",
                 color="#00e5ff", fontsize=10, pad=8)
    ax.legend(facecolor=PANEL, edgecolor="#546e7a", labelcolor="#e8eaf6", fontsize=8)
    ax.grid(True, axis="y", color="#1e293b", alpha=0.6)
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right", color="#e8eaf6", fontsize=8)

    # Right: Attention-vs-distance curve for key variants
    ax = axes[1]
    key_pes = ["prime_05", "hybrid_90z", "zeta", "sinusoidal", "random_irr"]
    for pe in key_pes:
        if pe not in resonance_results: continue
        abd = resonance_results[pe].get("attn_by_dist", [])
        if abd:
            xs = list(range(1, min(200, len(abd))))
            ys = abd[1:min(200, len(abd))]
            ax.plot(xs, ys, color=PE_COL.get(pe, "white"), label=pe, lw=1.5, alpha=0.85)
    # Mark prime multiples on x-axis
    for p in [2, 3, 5, 7, 11, 13]:
        for k in range(1, 200//p+1):
            ax.axvline(p*k, color="#b2ff59", alpha=0.04, lw=0.5)
    ax.set_xlabel("Distance (tokens)", color="#90a4ae")
    ax.set_ylabel("Mean attention weight", color="#90a4ae")
    ax.set_title("Attention vs Distance (first 200 tokens)\n(faint green lines = prime multiples)",
                 color="#00e5ff", fontsize=10, pad=8)
    ax.legend(facecolor=PANEL, edgecolor="#546e7a", labelcolor="#e8eaf6", fontsize=8)
    ax.grid(True, color="#1e293b", alpha=0.4)

    fig.tight_layout()
    fig.savefig(f"{CFG['results_dir']}/prime_resonance.png", dpi=150, facecolor=DARK)
    plt.close()
    print("  Saved prime_resonance.png")



In [ ]:
# ─── Cell 13: Summary and analysis ────────────────────────────────────────────

print("\n" + "═"*80)
print("PHASE 3 RESULTS SUMMARY")
print("═"*80)
print(f"\nModel: {CFG['n_layers']}L × d{CFG['d_model']} × {CFG['n_heads']}h")
print(f"Data:  WikiText-103 | Steps: {CFG['max_steps']} | Train ctx: {CFG['train_seq_len']}")
print(f"LitM context: {CFG['litm_context_len']} tokens")

# Perplexity table
eval_lens = CFG["eval_seq_lens"]
header = f"{'PE':<14} {'ValPPL':<9}" + "".join(f" {'PPL@'+str(sl//1024)+'K':<9}" for sl in eval_lens)
print("\n" + header)
print("-" * len(header))
for pe in CFG["pe_variants"]:
    final = all_results.get(pe, {}).get("final_val_ppl", float("nan"))
    row = f"{pe:<14} {final:<9.1f}"
    for sl in eval_lens:
        ppl = ppl_results.get(pe, {}).get(sl, float("nan"))
        row += f" {ppl:<9.1f}"
    print(row)

# LitM table
fracs = CFG["litm_positions"]
print(f"\nLost-in-the-Middle @ {CFG['litm_context_len']} tokens")
hdr2 = f"{'PE':<14}" + "".join(f" {'@'+str(int(f*100))+'%':<9}" for f in fracs)
print(hdr2)
print("-" * len(hdr2))
for pe in CFG["pe_variants"]:
    res = litm_results.get(pe, {})
    row = f"{pe:<14}" + "".join(f" {res.get(f, 0):<9.1%}" for f in fracs)
    mid = res.get(0.5, 0)
    row += "  ← KEY" if mid > 0 else ""
    print(row)

# Degradation ratio table — the core metric
print("\n" + "─"*60)
print("DEGRADATION RATIO: PPL change from 512 → longest eval context")
print(f"  (negative = improves with context — the key signal)\n")
eval_lens = CFG["eval_seq_lens"]
max_len = max(eval_lens)
print(f"  {'PE':<18} {'PPL@512':<10} {'PPL@'+str(max_len//1024)+'K':<10} {'Δ%':<10} Verdict")
print("  " + "-"*60)
for pe in CFG["pe_variants"]:
    p512 = ppl_results.get(pe, {}).get(512, float("nan"))
    pmax = ppl_results.get(pe, {}).get(max_len, float("nan"))
    if not (math.isnan(p512) or math.isnan(pmax) or p512 == 0):
        delta = (pmax / p512 - 1) * 100
        verdict = "IMPROVES ✅" if delta < -0.5 else "flat" if abs(delta) < 1 else f"degrades"
    else:
        delta, verdict = float("nan"), "no data"
    print(f"  {pe:<18} {p512:<10.1f} {pmax:<10.1f} {delta:<+10.1f} {verdict}")

# Rotary comparison — the key new question
print("\n" + "─"*60)
print("ROTARY COMPARISON: zeta_rope vs rope vs random_irr_rope")
for pe in ["rope", "zeta_rope", "random_irr_rope"]:
    p512 = ppl_results.get(pe, {}).get(512, float("nan"))
    pmax = ppl_results.get(pe, {}).get(max_len, float("nan"))
    delta = (pmax/p512-1)*100 if not math.isnan(p512) and p512>0 else float("nan")
    print(f"  {pe:<20} PPL@512={p512:.1f}  Δ={delta:+.1f}%")
print("  If zeta_rope Δ < rope Δ: zeta frequencies help within RoPE architecture")
print("  If zeta_rope Δ ≈ random_irr_rope Δ: architecture matters, not structure")

# Key question
print("\n" + "─"*60)
print("KEY QUESTION: Does ZetaPE outperform random_irr at LitM@50%?")
zeta_mid = litm_results.get("zeta", {}).get(0.5, float("nan"))
rand_mid  = litm_results.get("random_irr", {}).get(0.5, float("nan"))
if not math.isnan(zeta_mid) and not math.isnan(rand_mid):
    delta = zeta_mid - rand_mid
    verdict = "YES — number-theoretic structure helps" if delta > 0.02 else \
              "MARGINAL" if delta > 0 else "NO — structure not the differentiator"
    print(f"  zeta@50%: {zeta_mid:.1%}  |  random_irr@50%: {rand_mid:.1%}  |  Δ={delta:+.1%}")
    print(f"  Verdict: {verdict}")

# Save everything
final_out = {
    "config": {k: str(v) for k, v in CFG.items()},
    "training":     all_results,
    "ppl_by_len":   ppl_results,
    "litm":         litm_results,
}
with open(f"{CFG['results_dir']}/phase3_FINAL.json", "w") as f:
    json.dump(final_out, f, indent=2, default=str)
print(f"\nAll results → {CFG['results_dir']}/phase3_FINAL.json")

In [ ]:
# ─── Cell 14: Download ─────────────────────────────────────────────────────────

from google.colab import files
import glob

to_dl = (glob.glob(f"{CFG['results_dir']}/*.json") +
         glob.glob(f"{CFG['results_dir']}/*.png"))

print(f"Downloading {len(to_dl)} files...")
for f in sorted(to_dl):
    print(f"  {os.path.basename(f)}")
    files.download(f)

print("\n✅ Done.")